# Распределение данных Greenplum — 30 заданий

Распределение определяет, на каком primary-сегменте хранится строка. В модуле используются учебные копии в `m_razhin` и источник `greenplum_training.dist_source`. Сначала выполните задание самостоятельно, затем раскройте эталонное решение.

In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 100
%sql postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex

## Учебный источник `greenplum_training.dist_source`

Одна строка — синтетическое событие. Источник содержит ровно **100000 строк** и физически
распределён по `event_id`. Он специально включает хорошие и плохие кандидаты на ключ.

| Колонка | Тип | Смысл | Зачем в модуле |
|---|---|---|---|
|`event_id`|bigint|уникальный ID события|высокая cardinality и равномерный hash|
|`event_date`|date|день в диапазоне 90 дней|пример фильтра/partition-кандидата|
|`user_id`|bigint|один из 20000 пользователей|повторяющийся JOIN/GROUP key|
|`country`|text|одна из 8 стран|низкая cardinality, полезный JOIN key|
|`city`|text|один из 200 городов|средняя cardinality|
|`hot_key`|text|`HOT` у 80000 строк|намеренно сильный data skew|
|`nullable_key`|integer|каждая пятая строка NULL|влияние NULL на distribution|
|`payload`|text|строки разной длины|различие row skew и byte skew|

### Системное поле `gp_segment_id`

`gp_segment_id` не входит в DDL таблицы. Greenplum предоставляет его при чтении как
идентификатор сегмента, на котором физически находится строка. Поэтому:

- `segment_id` использовать нельзя — такой колонки нет;
- `gp_segment_id` можно выбрать и переименовать в `segment_id` в результате;
- `GROUP BY gp_segment_id` показывает количество строк на каждом непустом сегменте;
- в этом источнике ожидаются четыре группы, потому что кластер имеет четыре primary content.

Посмотреть схему до решения:

```sql
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'greenplum_training'
  AND table_name = 'dist_source'
ORDER BY ordinal_position;
```


## 1. Физическая модель

Coordinator не хранит пользовательские строки. Для hash distribution Greenplum вычисляет хеш ключа и выбирает content. `DISTRIBUTED RANDOMLY` распределяет без бизнес-ключа. `DISTRIBUTED REPLICATED` хранит полный набор на каждом primary и подходит только маленьким измерениям.

### Hash distribution

Для строки Greenplum хеширует distribution columns и сопоставляет результат одному content. Одинаковый ключ всегда попадает на один логический сегмент — это основа colocated JOIN. Хеш-функция не может исправить частоты: если 80% строк имеют один hot key, они окажутся вместе.

### Random policy

`DISTRIBUTED RANDOMLY` не связывает размещение с бизнес-колонкой и часто даёт приемлемый баланс массовой загрузки. Однако две random-таблицы не colocated: одинаковые JOIN-ключи могут находиться на разных сегментах, поэтому запросу потребуется Motion.

### Replicated policy

Полный справочник хранится на каждом primary. Большой факт соединяется с ним локально без пересылки. Цена — умножение места и работы загрузки на число сегментов. Репликация предназначена для небольших, относительно стабильных измерений, а не фактов.

## 2. Как выбирать ключ

Хороший ключ одновременно имеет высокую кардинальность, равномерную частоту, мало NULL, стабилен и совпадает с ключами крупных JOIN. Уникальный ключ часто равномерен, но не всегда оптимален для JOIN. Низкая кардинальность и hot key создают skew.

### Кардинальность и частота

`count(distinct key)` недостаточно. Профиль кандидата включает число строк, distinct, NULL, максимальную частоту и долю top-значения. Два ключа одинаковой кардинальности могут иметь совершенно разный skew.

### Составной ключ

Комбинация `(a,b)` повышает разнообразие, если `a` слабый. Но colocated JOIN потребует соединения по совместимой комбинации. Добавленная ради баланса колонка может устранить skew хранения и одновременно ухудшить главный JOIN.

### NULL

NULL участвует в distribution policy. Большое число NULL в единственной колонке ведёт себя как hot key. Перед выбором policy обязательно измеряйте NULL rate.

## 3. Skew

Сравнивайте строки и байты каждого сегмента. `max/avg` легко интерпретировать: 1 — идеальный баланс. Коэффициент вариации `100*stddev/avg` удобен для сравнения таблиц. Обязательно учитывайте сегменты с нулём строк.

### Метрики

Для counts `n_i` используют `max/avg`, размах `(max-min)/avg` и коэффициент вариации `100*stddev_pop/avg`. Идеальные значения — 1 и 0%. Универсального допустимого порога нет: важны объём и влияние на реальные запросы.

### Нулевые сегменты

`GROUP BY gp_segment_id` не показывает пустой content. Корректная диагностика начинает со списка четырёх primary content и делает LEFT JOIN к counts, иначе экстремальный skew будет недооценён.

### Строки и байты

Строки могут иметь разный размер из-за text, JSON и массивов. Ровные counts не гарантируют одинаковый I/O. Поэтому сравнивают и `pg_relation_size` на сегментах.

In [ ]:
%%sql
SELECT gp_segment_id,count(*) rows_count
FROM greenplum_training.dist_source
GROUP BY gp_segment_id ORDER BY 1;

## 4. Motion

`Redistribute Motion` пересылает строки по новому хешу; `Broadcast Motion` копирует небольшой набор; `Gather Motion` собирает результат. Colocated JOIN возможен, когда обе большие таблицы распределены совместимо по ключу соединения. Replicated dimension также устраняет пересылку факта.

### Redistribute

Строка получает новый destination content и может уйти по interconnect. Это требуется для JOIN, DISTINCT или GROUP BY по несовместимому ключу. Стоимость зависит от количества и ширины строк после ранних фильтров.

### Broadcast

Небольшой набор копируется каждому принимающему сегменту, а большой остаётся на месте. Ошибочная статистика может заставить optimizer broadcast-ить слишком большой набор.

### Gather

Результат собирается одному получателю. Небольшой финальный Gather нормален; ранний Gather большого набора уничтожает параллелизм.

### Colocation

Нужны совместимые типы, одинаковое число distribution columns и JOIN по всем ключам. Одинакового имени колонки недостаточно. Выражение или cast над ключом также может потребовать Motion.

### Чтение EXPLAIN

Читайте снизу вверх. Для каждого Motion спросите: какой набор движется, почему текущая policy не подходит, можно ли уменьшить строки раньше и достаточно ли част запрос, чтобы менять физическую модель.

In [ ]:
%%sql
EXPLAIN SELECT *
FROM greenplum_training.dist_source a
JOIN greenplum_training.dist_dimension b USING(country);

## 5. Runtime skew

Равномерное хранение не гарантирует равномерный запрос. Фильтр, JOIN или GROUP BY может оставить на одном сегменте намного больше строк. Поэтому анализируют и policy таблицы, и фактическое распределение промежуточного результата.

### Filter skew

Таблица по event_id может оставаться равномерной после фильтра страны. Таблица по country отправит все строки одной страны одному content. Это может уменьшить число работающих сегментов до одного.

### Join skew

При Redistribute частое значение JOIN-ключа собирается у одного получателя. Особенно опасны `unknown`, пустые значения и NULL. Остальные сегменты заканчивают раньше и ждут перегруженный.

### Aggregation skew

Частичная агрегация уменьшает сеть, но крупнейшая группа всё равно обрабатывается одним получателем финальной стадии. Отличайте локальный и финальный aggregate в плане.

## 6. Distribution и partitioning

Distribution отвечает «на каком сегменте строка», partitioning — «в какой физической части». Таблица может быть партиционирована по дате и распределена по visitid. Ключи решают разные задачи и не обязаны совпадать.

## 7. Порядок работы

1. Назовите гранулярность. 2. Измерьте cardinality, NULL и top frequency. 3. Перечислите крупные JOIN/GROUP BY. 4. Создайте варианты. 5. Посчитайте row/byte skew. 6. Сравните EXPLAIN и Motion. 7. Проверьте runtime skew. 8. Сформулируйте рекомендацию по измерениям.

### Задание 1. `m_razhin.gpd_01_segment_rows`

**Что сделать:** Исследуйте фактическое размещение всех 100000 строк `greenplum_training.dist_source` на четырёх primary-сегментах и сохраните результат в VIEW с колонками `segment_id`, `rows_count`.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

**Вход:** `greenplum_training.dist_source`, grain — одно событие, 100000 строк,
distribution policy — `DISTRIBUTED BY (event_id)`.

**Целевой результат:** VIEW `m_razhin.gpd_01_segment_rows` с двумя колонками:

| Колонка | Смысл |
|---|---|
|`segment_id`|значение системного `gp_segment_id`|
|`rows_count`|число строк источника на этом сегменте|

**Алгоритм без готового решения:**

1. Проверьте колонки источника через `information_schema.columns`.
2. Убедитесь, что `segment_id` среди них отсутствует.
3. Сгруппируйте строки по системному идентификатору физического сегмента.
4. Переименуйте этот идентификатор в `segment_id` в выходном VIEW.
5. Проверьте, что получены четыре строки, а сумма `rows_count` равна 100000.

**Что проверяет checker:** точное имя VIEW, четыре сегментные строки и сохранение всех
100000 исходных строк. Обычный `SELECT` не проходит: результат необходимо сохранить в VIEW.

**Типичные ошибки:** `segment_id` вместо `gp_segment_id`; `CREATE VIEW IF EXISTS`
(такого синтаксиса нет); пропущенное `AS`; выполнение SELECT без создания VIEW.


<details><summary>Подсказка</summary>

Физический сегмент определяется системным полем `gp_segment_id`; обычной колонки `segment_id` в источнике нет.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_01_segment_rows;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_01_segment_rows LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_01_segment_rows;
CREATE VIEW m_razhin.gpd_01_segment_rows AS
SELECT gp_segment_id AS segment_id,count(*) AS rows_count
FROM greenplum_training.dist_source
GROUP BY gp_segment_id;
```

**Почему так:** Системное поле gp_segment_id показывает primary-сегмент; одна строка результата — один сегмент.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',1);

### Задание 2. `m_razhin.gpd_02_random`

**Что сделать:** Создайте таблицу-копию источника `DISTRIBUTED RANDOMLY`.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

CTAS поддерживает предложение распределения.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_02_random;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_02_random LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP TABLE IF EXISTS m_razhin.gpd_02_random;
CREATE TABLE m_razhin.gpd_02_random AS
SELECT * FROM greenplum_training.dist_source
DISTRIBUTED RANDOMLY;
```

**Почему так:** CTAS копирует 100000 событий, а RANDOMLY не связывает размещение с бизнес-ключом.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',2);

### Задание 3. `m_razhin.gpd_03_by_event`

**Что сделать:** Создайте копию `DISTRIBUTED BY (event_id)`.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Уникальный равномерный ключ — базовый вариант.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_03_by_event;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_03_by_event LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP TABLE IF EXISTS m_razhin.gpd_03_by_event;
CREATE TABLE m_razhin.gpd_03_by_event AS
SELECT * FROM greenplum_training.dist_source
DISTRIBUTED BY (event_id);
```

**Почему так:** Уникальный event_id даёт высокую кардинальность и обычно равномерный hash distribution.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',3);

### Задание 4. `m_razhin.gpd_04_by_country`

**Что сделать:** Создайте копию `DISTRIBUTED BY (country)`.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Низкая кардинальность может дать перекос.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_04_by_country;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_04_by_country LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP TABLE IF EXISTS m_razhin.gpd_04_by_country;
CREATE TABLE m_razhin.gpd_04_by_country AS
SELECT * FROM greenplum_training.dist_source
DISTRIBUTED BY (country);
```

**Почему так:** Всего восемь значений country, поэтому ключ полезен для JOIN, но слабее для баланса.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',4);

### Задание 5. `m_razhin.gpd_05_by_hot_key`

**Что сделать:** Создайте копию `DISTRIBUTED BY (hot_key)`.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Источник специально содержит одно очень частое значение.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_05_by_hot_key;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_05_by_hot_key LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP TABLE IF EXISTS m_razhin.gpd_05_by_hot_key;
CREATE TABLE m_razhin.gpd_05_by_hot_key AS
SELECT * FROM greenplum_training.dist_source
DISTRIBUTED BY (hot_key);
```

**Почему так:** Значение HOT встречается у 80% строк и намеренно создаёт сильный перекос.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',5);

### Задание 6. `m_razhin.gpd_06_distribution`

**Что сделать:** Создайте VIEW segment_id, rows_count для таблицы gpd_05_by_hot_key.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Сравните максимум и среднее.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_06_distribution;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_06_distribution LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_06_distribution;
CREATE VIEW m_razhin.gpd_06_distribution AS
WITH segments AS (
 SELECT content AS segment_id FROM gp_segment_configuration
 WHERE role='p' AND content>=0
), counts AS (
 SELECT gp_segment_id AS segment_id,count(*) AS rows_count
 FROM m_razhin.gpd_05_by_hot_key GROUP BY gp_segment_id
)
SELECT s.segment_id,coalesce(c.rows_count,0) AS rows_count
FROM segments s LEFT JOIN counts c USING(segment_id);
```

**Почему так:** Список primary content сохраняет даже пустой сегмент, поэтому профиль skew не занижен.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',6);

### Задание 7. `m_razhin.gpd_07_skew_ratio`

**Что сделать:** Создайте VIEW с одной колонкой skew_ratio = max(rows)/avg(rows), округление 4 знака.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Не теряйте сегменты с нулём строк.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_07_skew_ratio;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_07_skew_ratio LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_07_skew_ratio;
CREATE VIEW m_razhin.gpd_07_skew_ratio AS
WITH counts AS (
 SELECT count(*)::numeric AS rows_count
 FROM m_razhin.gpd_05_by_hot_key GROUP BY gp_segment_id
)
SELECT round(max(rows_count)/avg(rows_count),4) AS skew_ratio FROM counts;
```

**Почему так:** max/avg равен 1 при идеальном балансе и растёт вместе с перегрузкой крупнейшего сегмента.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',7);

### Задание 8. `m_razhin.gpd_08_skew_coefficient`

**Что сделать:** Создайте VIEW с `skew_coefficient` в процентах для gpd_05.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Используйте стандартное отклонение относительно среднего.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_08_skew_coefficient;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_08_skew_coefficient LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_08_skew_coefficient;
CREATE VIEW m_razhin.gpd_08_skew_coefficient AS
WITH counts AS (
 SELECT count(*)::numeric AS rows_count
 FROM m_razhin.gpd_05_by_hot_key GROUP BY gp_segment_id
)
SELECT round(100*stddev_pop(rows_count)/nullif(avg(rows_count),0),4) AS skew_coefficient
FROM counts;
```

**Почему так:** Коэффициент вариации выражает стандартное отклонение counts в процентах от среднего.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',8);

### Задание 9. `m_razhin.gpd_09_null_key`

**Что сделать:** Создайте таблицу по nullable_key и исследуйте размещение NULL.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

NULL тоже участвует в хешировании.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_09_null_key;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_09_null_key LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP TABLE IF EXISTS m_razhin.gpd_09_null_key;
CREATE TABLE m_razhin.gpd_09_null_key AS
SELECT * FROM greenplum_training.dist_source
DISTRIBUTED BY (nullable_key);
```

**Почему так:** Все NULL одного distribution key хешируются совместимо и могут вести себя как частое значение.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',9);

### Задание 10. `m_razhin.gpd_10_composite`

**Что сделать:** Создайте копию `DISTRIBUTED BY (country, user_id)`.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Составной ключ может компенсировать слабую первую колонку.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_10_composite;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_10_composite LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP TABLE IF EXISTS m_razhin.gpd_10_composite;
CREATE TABLE m_razhin.gpd_10_composite AS
SELECT * FROM greenplum_training.dist_source
DISTRIBUTED BY (country,user_id);
```

**Почему так:** user_id повышает разнообразие комбинации и компенсирует низкую кардинальность country.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',10);

## Уровень 2 — JOIN и Motion

### Задание 11. `m_razhin.gpd_11_replicated_dim`

**Что сделать:** Создайте `m_razhin.gpd_11_replicated_dim` из справочника стран с replicated policy.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Репликация подходит только малым справочникам.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_11_replicated_dim;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_11_replicated_dim LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP TABLE IF EXISTS m_razhin.gpd_11_replicated_dim;
CREATE TABLE m_razhin.gpd_11_replicated_dim AS
SELECT country AS country_code,country_name,region
FROM greenplum_training.dist_dimension
DISTRIBUTED REPLICATED;
```

**Почему так:** Восемь строк справочника копируются на каждый primary и доступны локально для любого факта.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',11);

### Задание 12. `m_razhin.gpd_12_dim_hash`

**Что сделать:** Создайте тот же справочник `DISTRIBUTED BY (country_code)`.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Он понадобится для сравнения планов.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_12_dim_hash;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_12_dim_hash LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP TABLE IF EXISTS m_razhin.gpd_12_dim_hash;
CREATE TABLE m_razhin.gpd_12_dim_hash AS
SELECT country AS country_code,country_name,region
FROM greenplum_training.dist_dimension
DISTRIBUTED BY (country_code);
```

**Почему так:** Hash-вариант нужен для colocated JOIN с фактом, распределённым совместимо по country.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',12);

### Задание 13. `m_razhin.gpd_13_fact_country`

**Что сделать:** Создайте fact-копию, распределённую по country.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Ключ должен совпасть с ключом JOIN.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_13_fact_country;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_13_fact_country LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP TABLE IF EXISTS m_razhin.gpd_13_fact_country;
CREATE TABLE m_razhin.gpd_13_fact_country AS
SELECT * FROM greenplum_training.dist_source
DISTRIBUTED BY (country);
```

**Почему так:** Distribution key факта совпадает с ключом будущего JOIN со справочником.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',13);

### Задание 14. `m_razhin.gpd_14_colocated_join`

**Что сделать:** Создайте VIEW результата JOIN gpd_13_fact_country и hash-справочника по стране.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Физические типы и distribution keys должны совпадать.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_14_colocated_join;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_14_colocated_join LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_14_colocated_join;
CREATE VIEW m_razhin.gpd_14_colocated_join AS
SELECT f.*,d.country_name,d.region
FROM m_razhin.gpd_13_fact_country f
JOIN m_razhin.gpd_12_dim_hash d ON d.country_code=f.country;
```

**Почему так:** Обе стороны хешируются по совместимым текстовым ключам, поэтому совпадающие строки уже colocated.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',14);

### Задание 15. `m_razhin.gpd_15_replicated_join`

**Что сделать:** Создайте VIEW JOIN факта с replicated-справочником.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Реплицированный набор локален на каждом сегменте.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_15_replicated_join;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_15_replicated_join LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_15_replicated_join;
CREATE VIEW m_razhin.gpd_15_replicated_join AS
SELECT f.*,d.country_name,d.region
FROM greenplum_training.dist_source f
JOIN m_razhin.gpd_11_replicated_dim d ON d.country_code=f.country;
```

**Почему так:** Реплицированная сторона присутствует на каждом сегменте и не требует пересылки большого факта.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',15);

### Задание 16. `m_razhin.gpd_16_motion_plan`

**Что сделать:** Сохраните в VIEW количество Motion-строк плана несовместимого JOIN.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Используйте EXPLAIN через учебную функцию.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_16_motion_plan;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_16_motion_plan LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_16_motion_plan;
CREATE VIEW m_razhin.gpd_16_motion_plan AS
SELECT greenplum_training.motion_count(
 'SELECT * FROM greenplum_training.dist_source a JOIN greenplum_training.dist_dimension b USING(country)'
) AS motion_count;
```

**Почему так:** Учебная функция выполняет EXPLAIN и считает строки плана, содержащие Motion.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',16);

### Задание 17. `m_razhin.gpd_17_colocated_plan`

**Что сделать:** Сохраните количество Motion для colocated JOIN.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Сравните с предыдущим заданием.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_17_colocated_plan;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_17_colocated_plan LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_17_colocated_plan;
CREATE VIEW m_razhin.gpd_17_colocated_plan AS
SELECT greenplum_training.motion_count(
 'SELECT * FROM m_razhin.gpd_13_fact_country f JOIN m_razhin.gpd_12_dim_hash d ON d.country_code=f.country'
) AS motion_count;
```

**Почему так:** Совместимые policies должны дать не больше Motion, чем исходный несовместимый вариант.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',17);

### Задание 18. `m_razhin.gpd_18_random_join`

**Что сделать:** Создайте random fact и измерьте Motion при JOIN.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Random policy не гарантирует colocated строки.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_18_random_join;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_18_random_join LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP TABLE IF EXISTS m_razhin.gpd_18_random_join;
CREATE TABLE m_razhin.gpd_18_random_join AS
SELECT * FROM greenplum_training.dist_source
DISTRIBUTED RANDOMLY;

SELECT greenplum_training.motion_count(
 'SELECT * FROM m_razhin.gpd_18_random_join f JOIN greenplum_training.dist_dimension d USING(country)'
) AS motion_count;
```

**Почему так:** Random policy может быть ровной, но не гарантирует совместного размещения одинаковых country.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',18);

### Задание 19. `m_razhin.gpd_19_group_motion`

**Что сделать:** Измерьте Motion для GROUP BY country на таблице, распределённой по event_id.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Агрегат по чужому ключу требует обмена.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_19_group_motion;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_19_group_motion LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_19_group_motion;
CREATE VIEW m_razhin.gpd_19_group_motion AS
SELECT greenplum_training.motion_count(
 'SELECT country,count(*) FROM greenplum_training.dist_source GROUP BY country'
) AS motion_count;
```

**Почему так:** Источник распределён по event_id, поэтому финальному GROUP BY country обычно нужен обмен строками.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',19);

### Задание 20. `m_razhin.gpd_20_local_group`

**Что сделать:** Измерьте Motion для GROUP BY country на таблице, распределённой по country.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Часть агрегации может быть локальной.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_20_local_group;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_20_local_group LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_20_local_group;
CREATE VIEW m_razhin.gpd_20_local_group AS
SELECT greenplum_training.motion_count(
 'SELECT country,count(*) FROM m_razhin.gpd_13_fact_country GROUP BY country'
) AS motion_count;
```

**Почему так:** Когда GROUP BY содержит distribution key, группы можно вычислять локально без перераспределения факта.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',20);

## Уровень 3 — runtime skew и проектирование Метрики

### Задание 21. `m_razhin.gpd_21_filter_skew`

**Что сделать:** Покажите распределение строк после фильтра `country='RU'`.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Даже ровная таблица может дать runtime skew после фильтра.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_21_filter_skew;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_21_filter_skew LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_21_filter_skew;
CREATE VIEW m_razhin.gpd_21_filter_skew AS
SELECT gp_segment_id AS segment_id,count(*) AS rows_count
FROM m_razhin.gpd_13_fact_country
WHERE country='RU'
GROUP BY gp_segment_id;
```

**Почему так:** Таблица распределена по country, поэтому фильтр одного значения оставляет работу только его сегменту.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',21);

### Задание 22. `m_razhin.gpd_22_join_skew`

**Что сделать:** Покажите распределение результата JOIN по hot_key.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Skew бывает не только у хранения, но и у операции.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_22_join_skew;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_22_join_skew LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_22_join_skew;
CREATE VIEW m_razhin.gpd_22_join_skew AS
SELECT f.gp_segment_id AS segment_id,count(*) AS rows_count
FROM m_razhin.gpd_05_by_hot_key f
JOIN (SELECT 'HOT'::text AS hot_key) d USING(hot_key)
GROUP BY f.gp_segment_id;
```

**Почему так:** JOIN сохраняет частое значение HOT; профиль результата показывает runtime skew принимающей операции.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',22);

### Задание 23. `m_razhin.gpd_23_size_by_segment`

**Что сделать:** Создайте VIEW размера gpd_05 на каждом сегменте.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Используйте `pg_relation_size` на сегментах.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_23_size_by_segment;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_23_size_by_segment LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_23_size_by_segment;
CREATE VIEW m_razhin.gpd_23_size_by_segment AS
SELECT gp_segment_id AS segment_id,
       pg_relation_size('m_razhin.gpd_05_by_hot_key'::regclass) AS size_bytes
FROM gp_dist_random('gp_id');
```

**Почему так:** gp_dist_random выполняет запрос системного отношения на каждом primary и возвращает локальный размер файла.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',23);

### Задание 24. `m_razhin.gpd_24_rows_vs_bytes`

**Что сделать:** Создайте VIEW, сравнивающее долю строк и долю байтов сегмента.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Одинаковое число строк не гарантирует одинаковый объём.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_24_rows_vs_bytes;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_24_rows_vs_bytes LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_24_rows_vs_bytes;
CREATE VIEW m_razhin.gpd_24_rows_vs_bytes AS
WITH rows_by_segment AS (
 SELECT segment_id,rows_count::numeric FROM m_razhin.gpd_06_distribution
), bytes_by_segment AS (
 SELECT segment_id,size_bytes::numeric FROM m_razhin.gpd_23_size_by_segment
)
SELECT r.segment_id,r.rows_count,b.size_bytes,
       r.rows_count/nullif(sum(r.rows_count) OVER(),0) AS row_share,
       b.size_bytes/nullif(sum(b.size_bytes) OVER(),0) AS byte_share
FROM rows_by_segment r JOIN bytes_by_segment b USING(segment_id);
```

**Почему так:** Обе доли нормируются на общий итог; различие показывает влияние переменной длины payload.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',24);

### Задание 25. `m_razhin.gpd_25_metrica_profile`

**Что сделать:** Создайте профиль кардинальности date, regioncountry, regioncity, ipaddress источника Метрики.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Верните количество строк и distinct для каждого кандидата.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_25_metrica_profile;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_25_metrica_profile LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_25_metrica_profile;
CREATE VIEW m_razhin.gpd_25_metrica_profile AS
WITH profile AS (
 SELECT count(*) AS total_rows,
        count(DISTINCT date) AS date_ndv,
        count(DISTINCT regioncountry) AS country_ndv,
        count(DISTINCT regioncity) AS city_ndv,
        count(DISTINCT ipaddress) AS ip_ndv
 FROM dds.ext_raw_yndx_metrica_logs
)
SELECT 'date'::text AS column_name,total_rows,date_ndv AS distinct_values FROM profile
UNION ALL SELECT 'regioncountry',total_rows,country_ndv FROM profile
UNION ALL SELECT 'regioncity',total_rows,city_ndv FROM profile
UNION ALL SELECT 'ipaddress',total_rows,ip_ndv FROM profile;
```

**Почему так:** Один профильный scan вычисляет total rows и NDV всех четырёх кандидатов, после чего показатели разворачиваются в строки.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',25);

### Задание 26. `m_razhin.gpd_26_metrica_candidate`

**Что сделать:** Создайте учебную выборку Метрики с выбранным ключом распределения.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Не изменяйте исходную внешнюю таблицу.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_26_metrica_candidate;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_26_metrica_candidate LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP TABLE IF EXISTS m_razhin.gpd_26_metrica_candidate;
CREATE TABLE m_razhin.gpd_26_metrica_candidate AS
SELECT date,dt,visitid,isnewuser,starturl,endurl,pageviews,visitduration,
       regioncountry,regioncity,clientid,ipaddress,clienttimezone,
       devicecategory,mobilephone,mobilephonemodel,operatingsystem,browser
FROM dds.ext_raw_yndx_metrica_logs
LIMIT 100000
DISTRIBUTED BY (visitid);
```

**Почему так:** visitid имеет высокую кардинальность и выбран для баланса; поля регулярных фильтров сохраняются колонками.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',26);

### Задание 27. `m_razhin.gpd_27_metrica_skew`

**Что сделать:** Рассчитайте распределение и skew_ratio созданной выборки.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Обоснуйте ключ в markdown перед SQL.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_27_metrica_skew;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_27_metrica_skew LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_27_metrica_skew;
CREATE VIEW m_razhin.gpd_27_metrica_skew AS
WITH segments AS (
 SELECT content AS segment_id FROM gp_segment_configuration WHERE role='p' AND content>=0
), counts AS (
 SELECT gp_segment_id AS segment_id,count(*)::numeric AS rows_count
 FROM m_razhin.gpd_26_metrica_candidate GROUP BY gp_segment_id
), full_counts AS (
 SELECT s.segment_id,coalesce(c.rows_count,0) AS rows_count
 FROM segments s LEFT JOIN counts c USING(segment_id)
)
SELECT count(*)::int AS segment_count,
       round(max(rows_count)/nullif(avg(rows_count),0),4) AS skew_ratio
FROM full_counts;
```

**Почему так:** Четыре primary учитываются даже при нулевых counts; max/avg измеряет фактический баланс выборки.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',27);

### Задание 28. `m_razhin.gpd_28_query_motion`

**Что сделать:** Проверьте Motion для типового фильтра Метрики из задания.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Фильтр сам по себе не обязан создавать Motion.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_28_query_motion;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_28_query_motion LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_28_query_motion;
CREATE VIEW m_razhin.gpd_28_query_motion AS
SELECT greenplum_training.motion_count(
 $$SELECT dt,visitid,endurl,visitduration,clientid
   FROM m_razhin.gpd_26_metrica_candidate
   WHERE date='2025-05-08' AND regioncountry='Russia'
     AND regioncity='Moscow' AND ipaddress='178.176.79.xxx'$$
) AS motion_count;
```

**Почему так:** Обычный фильтр выполняется локально на сегментах; функция фиксирует реальный Motion выбранного плана.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',28);

### Задание 29. `m_razhin.gpd_29_redistribute`

**Что сделать:** Создайте новую копию источника с улучшенной policy относительно gpd_05.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

CTAS безопаснее учебного изменения исходника.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_29_redistribute;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_29_redistribute LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP TABLE IF EXISTS m_razhin.gpd_29_redistribute;
CREATE TABLE m_razhin.gpd_29_redistribute AS
SELECT * FROM greenplum_training.dist_source
DISTRIBUTED BY (event_id);
```

**Почему так:** Новая CTAS-копия устраняет hot_key policy, не изменяя исходную учебную таблицу.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',29);

### Задание 30. `m_razhin.gpd_30_recommendation`

**Что сделать:** Создайте VIEW-рекомендацию: table_name, policy, skew_ratio, motion_count, verdict для трёх вариантов.

Перед созданием объекта зафиксируйте ожидаемое распределение или план. После создания сравните ожидание с измерением.

<details><summary>Подсказка</summary>

Итог должен опираться на измерения.

</details>

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- DROP VIEW/TABLE IF EXISTS m_razhin.gpd_30_recommendation;
-- Создайте требуемый объект здесь.

In [ ]:
%%sql
-- Ручная проверка
-- SELECT * FROM m_razhin.gpd_30_recommendation LIMIT 20;

<details><summary><strong>Эталонное решение и разбор</strong></summary>

```sql
DROP VIEW IF EXISTS m_razhin.gpd_30_recommendation;
CREATE VIEW m_razhin.gpd_30_recommendation AS
WITH variants AS (
 SELECT 'gpd_03_by_event'::text AS table_name,
        pg_get_table_distributedby('m_razhin.gpd_03_by_event'::regclass) AS policy,
        (SELECT max(c)::numeric/avg(c) FROM (SELECT count(*) c FROM m_razhin.gpd_03_by_event GROUP BY gp_segment_id)s) AS skew_ratio,
        greenplum_training.motion_count('SELECT * FROM m_razhin.gpd_03_by_event f JOIN greenplum_training.dist_dimension d USING(country)') AS motion_count,
        'лучший баланс'::text AS verdict
 UNION ALL
 SELECT 'gpd_04_by_country',pg_get_table_distributedby('m_razhin.gpd_04_by_country'::regclass),
        (SELECT max(c)::numeric/avg(c) FROM (SELECT count(*) c FROM m_razhin.gpd_04_by_country GROUP BY gp_segment_id)s),
        greenplum_training.motion_count('SELECT * FROM m_razhin.gpd_04_by_country f JOIN greenplum_training.dist_dimension d USING(country)'),
        'colocated JOIN, проверить skew'
 UNION ALL
 SELECT 'gpd_05_by_hot_key',pg_get_table_distributedby('m_razhin.gpd_05_by_hot_key'::regclass),
        (SELECT max(c)::numeric/avg(c) FROM (SELECT count(*) c FROM m_razhin.gpd_05_by_hot_key GROUP BY gp_segment_id)s),
        greenplum_training.motion_count('SELECT * FROM m_razhin.gpd_05_by_hot_key f JOIN greenplum_training.dist_dimension d USING(country)'),
        'не использовать: hot key'
)
SELECT table_name,policy,round(skew_ratio,4) AS skew_ratio,motion_count,verdict FROM variants;
```

**Почему так:** Итоговая рекомендация сопоставляет policy, измеренный skew и Motion трёх физических вариантов.

После выполнения сравните policy, сумму строк по сегментам и коэффициент skew.

</details>

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('distribution',30);

## Прогресс

In [ ]:
%%sql
SELECT * FROM greenplum_training.progress WHERE module_name='distribution' ORDER BY task_no;